# Práctica 11 — Regresión logística y modelos lineales generalizados

**Curso:** Estadística Aplicada con Python · **Bloque 4:** Modelos y experimentos · **Tiempo estimado:** 60 minutos
**Clase asociada:** `notebooks_clase/clase11_regresion_logistica_glm.ipynb`

## Cómo funciona esta práctica

- Cada ejercicio tiene una celda con `# TODO`: complétala.
- Al final de cada ejercicio hay una **comprobación automática** (`assert`). Si falla, el mensaje te dice qué revisar. Si olvidas completar un `TODO`, el mensaje te lo recuerda.
- Los ejercicios con simulación o con datos aleatorios usan tolerancias basadas en el error estándar, así que un resultado correcto no debería fallar por casualidad.
- Las **soluciones** están al final del notebook, en una sección aparte. **No las abras antes de haberlo intentado.**
- Ejecuta las celdas en orden. Todos los datos son sintéticos: *ejemplos ilustrativos*.

---
## Celda 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# Coeficientes VERDADEROS con los que generamos los datos (solo los conocemos porque son sintéticos)
VERDAD = {"Intercept": -1.8, "antiguedad": -0.03, "gasto": 0.03, "datos_gb": 0.0, "tickets": 0.45, "plan_anual": -0.9}

def generar_clientes(n, rng):
    """Genera n clientes sintéticos (ejemplo ilustrativo) con abandono según VERDAD."""
    antiguedad = np.clip(rng.gamma(2.0, 14, n), 1, 72).round()
    gasto = np.clip(30 + 0.25 * antiguedad + rng.normal(0, 10, n), 10, None).round(1)
    datos_gb = np.clip(0.09 * gasto + rng.normal(0, 0.25, n), 0.1, None).round(2)   # la usaremos en la clase 12
    tickets = rng.poisson(np.exp(0.008 * (gasto - 40)))
    plan_anual = rng.binomial(1, 0.35, n)
    eta = (VERDAD["Intercept"] + VERDAD["antiguedad"] * antiguedad + VERDAD["gasto"] * gasto
           + VERDAD["tickets"] * tickets + VERDAD["plan_anual"] * plan_anual)
    abandona = rng.binomial(1, 1 / (1 + np.exp(-eta)))
    return pd.DataFrame({"abandona": abandona, "antiguedad": antiguedad, "gasto": gasto,
                         "datos_gb": datos_gb, "tickets": tickets, "plan_anual": plan_anual})

print("Librerías y funciones cargadas correctamente.")

## Ejercicio 1 — De probabilidad a odds, log-odds y odds ratio

Con el array `p` de probabilidades calcula: `odds` (p / (1 − p)), `logodds` (su logaritmo natural) y `p_de_vuelta` (la probabilidad recuperada desde los log-odds con la sigmoide). Después, un coeficiente `beta = 0.69` da `razon_odds = exp(beta)`: aplícalo a un cliente con `p0 = 0.20` (multiplica sus odds por `razon_odds`) y calcula `p_nueva`, su nueva probabilidad. Por último, `prob_se_duplica` debe ser `True` si `p_nueva` es el doble de `p0` (aproximadamente) y `False` si no.

In [ ]:
p = np.array([0.05, 0.20, 0.50, 0.90])
beta = 0.69
p0 = 0.20

odds = None              # TODO
logodds = None           # TODO
p_de_vuelta = None       # TODO: la sigmoide 1 / (1 + exp(-logodds))
razon_odds = None        # TODO
p_nueva = None           # TODO: odds de p0 × razon_odds, y de ahí la probabilidad
prob_se_duplica = None   # TODO

# Comprobación automática
assert all(v is not None for v in (odds, logodds, p_de_vuelta, razon_odds, p_nueva, prob_se_duplica)), "Completa las líneas marcadas con TODO"
assert np.allclose(odds, p / (1 - p)), "odds = p / (1 - p)"
assert np.allclose(logodds, np.log(p / (1 - p))), "los log-odds son el logaritmo natural de los odds"
assert np.allclose(p_de_vuelta, p), "la sigmoide de los log-odds debe devolver p"
assert np.isclose(razon_odds, np.exp(0.69)), "razon_odds = exp(beta)"
assert np.isclose(p_nueva, 1 / 3, atol=0.005), "odds de p0 = 0,25; nuevos odds = 0,25 × razon_odds ≈ 0,5; p = odds / (1 + odds)"
assert prob_se_duplica is False, "los odds se duplican, pero la probabilidad pasa de 0,20 a 0,333, no a 0,40"
print(f"odds = {np.round(odds, 3)}   p_nueva = {p_nueva:.3f}")
print("Ejercicio 1 correcto ✔")

## Ejercicio 2 — Ajustar, odds ratio e intervalo

Genera 1 500 clientes con `generar_clientes` y ajusta con `smf.logit` el modelo de la variable `formula` (sin `datos_gb`). Calcula:

- `modelo`: el resultado de `.fit(disp=0)`,
- `odds_ratios`: `np.exp(modelo.params)`,
- `ic_or`: `np.exp(modelo.conf_int())` (DataFrame con dos columnas: 0 y 1),
- `or_tickets_10pct`: el aumento porcentual de los **odds** por cada ticket, `(OR − 1) × 100`.

In [ ]:
datos = generar_clientes(1500, np.random.default_rng(2026))
formula = "abandona ~ antiguedad + gasto + tickets + plan_anual"

modelo = None             # TODO: smf.logit(formula, data=datos).fit(disp=0)
odds_ratios = None        # TODO
ic_or = None              # TODO
or_tickets_10pct = None   # TODO

# Comprobación automática
assert all(v is not None for v in (modelo, odds_ratios, ic_or, or_tickets_10pct)), "Completa las líneas marcadas con TODO"
ref = smf.logit(formula, data=datos).fit(disp=0)
assert np.allclose(modelo.params, ref.params), "ajusta smf.logit con la fórmula y los datos indicados"
assert np.allclose(odds_ratios, np.exp(ref.params)), "odds ratio = exp de los coeficientes"
assert np.allclose(ic_or.values, np.exp(ref.conf_int()).values), "aplica exp a los extremos del intervalo, no al ancho"
assert np.isclose(or_tickets_10pct, (np.exp(ref.params["tickets"]) - 1) * 100), "(exp(coef de tickets) − 1) × 100"
for v in ("tickets", "plan_anual", "antiguedad"):
    assert abs(modelo.params[v] - VERDAD[v]) < 4 * modelo.bse[v], f"el coeficiente estimado de {v} debería estar cerca del verdadero"
print(f"Cada ticket multiplica los odds por {odds_ratios['tickets']:.2f} (+{or_tickets_10pct:.0f} %)")
print("Ejercicio 2 correcto ✔")

## Ejercicio 3 — Probabilidades previstas y efecto marginal

Con `datos` (del ejercicio 2), el modelo `modelo3` ya está ajustado. Hay dos perfiles: `perfil_medio` (24 meses, 40 €, 1 ticket, sin plan anual) y `perfil_bajo` (72 meses, 20 €, 0 tickets, con plan anual). Con la función `con_un_ticket_mas` calcula `cambio_medio` y `cambio_bajo`: el cambio de probabilidad prevista (en **puntos porcentuales**, un número) al añadir un ticket a cada perfil. Calcula también `ame_pp`: el efecto marginal medio de un ticket más sobre **todos** los clientes de `datos`, en puntos porcentuales. Por último, `mayor_en_medio` es `True` si el cambio del perfil medio es mayor que el del perfil bajo (el mismo odds ratio, distinto efecto).

In [ ]:
modelo3 = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=datos).fit(disp=0)
perfil_medio = pd.DataFrame({"antiguedad": [24], "gasto": [40], "tickets": [1], "plan_anual": [0]})
perfil_bajo = pd.DataFrame({"antiguedad": [72], "gasto": [20], "tickets": [0], "plan_anual": [1]})

def con_un_ticket_mas(perfil):
    nuevo = perfil.copy()
    nuevo["tickets"] = nuevo["tickets"] + 1
    return nuevo

cambio_medio = None     # TODO: (modelo3.predict(con_un_ticket_mas(perfil_medio)) − modelo3.predict(perfil_medio)) × 100, como número
cambio_bajo = None      # TODO
ame_pp = None           # TODO: promedio, sobre todas las filas de datos, del cambio de probabilidad al sumar 1 a tickets
mayor_en_medio = None   # TODO

# Comprobación automática
assert all(v is not None for v in (cambio_medio, cambio_bajo, ame_pp, mayor_en_medio)), "Completa las líneas marcadas con TODO"
ref_m = float((modelo3.predict(con_un_ticket_mas(perfil_medio)) - modelo3.predict(perfil_medio)).iloc[0] * 100)
ref_b = float((modelo3.predict(con_un_ticket_mas(perfil_bajo)) - modelo3.predict(perfil_bajo)).iloc[0] * 100)
ref_ame = float((modelo3.predict(con_un_ticket_mas(datos)) - modelo3.predict(datos)).mean() * 100)
assert np.isclose(np.squeeze(np.asarray(cambio_medio, dtype=float)), ref_m), "el cambio se mide en puntos porcentuales: (p después − p antes) × 100"
assert np.isclose(np.squeeze(np.asarray(cambio_bajo, dtype=float)), ref_b), "usa perfil_bajo con la función con_un_ticket_mas"
assert np.isclose(ame_pp, ref_ame), "el AME promedia el cambio de probabilidad de CADA cliente de datos"
assert bool(mayor_en_medio) is True, "compara cambio_medio con cambio_bajo: la probabilidad de partida baja deja poco margen"
print(f"Cambio con +1 ticket: perfil medio {ref_m:.1f} puntos, perfil bajo {ref_b:.1f} puntos, AME {ref_ame:.1f} puntos")
print("Ejercicio 3 correcto ✔")

## Ejercicio 4 — Variable categórica: dummies y lectura del odds ratio

Con `datos` crea la columna `tramo` agrupando `antiguedad` (hasta 11 meses `"nuevo"`, de 12 a 35 `"medio"`, 36 o más `"veterano"`). Ajusta `m_tramo` con `smf.logit`, usando `"nuevo"` como categoría de referencia y las variables `gasto`, `tickets` y `plan_anual`. Calcula `or_veterano` (odds ratio de `veterano` frente a `nuevo`), `pct_menos_odds` (en qué porcentaje reduce los odds, `(1 − OR) × 100`) y `or_vet_vs_medio` (odds ratio de `veterano` frente a `medio`: `exp(b_veterano − b_medio)`).

In [ ]:
datos["tramo"] = pd.cut(datos["antiguedad"], [0, 11, 35, 72], labels=["nuevo", "medio", "veterano"]).astype(str)
f_tramo = "abandona ~ C(tramo, Treatment(reference='nuevo')) + gasto + tickets + plan_anual"
n_vet = "C(tramo, Treatment(reference='nuevo'))[T.veterano]"
n_med = "C(tramo, Treatment(reference='nuevo'))[T.medio]"

m_tramo = None          # TODO: smf.logit(f_tramo, data=datos).fit(disp=0)
or_veterano = None      # TODO: exp del coeficiente n_vet
pct_menos_odds = None   # TODO
or_vet_vs_medio = None  # TODO

# Comprobación automática
assert all(v is not None for v in (m_tramo, or_veterano, pct_menos_odds, or_vet_vs_medio)), "Completa las líneas marcadas con TODO"
ref = smf.logit(f_tramo, data=datos).fit(disp=0)
assert np.allclose(m_tramo.params, ref.params), "ajusta smf.logit con f_tramo y los datos"
assert np.isclose(or_veterano, np.exp(ref.params[n_vet])), "or_veterano = np.exp(m_tramo.params[n_vet])"
assert or_veterano < 1, "los clientes veteranos abandonan menos que los nuevos: el odds ratio es menor que 1"
assert np.isclose(pct_menos_odds, (1 - np.exp(ref.params[n_vet])) * 100), "(1 − odds ratio) × 100"
assert np.isclose(or_vet_vs_medio, np.exp(ref.params[n_vet] - ref.params[n_med])), "veterano frente a medio: exp(b_veterano − b_medio), o el cociente de los dos odds ratio"
print(f"Odds ratio veterano frente a nuevo: {or_veterano:.2f} (−{pct_menos_odds:.0f} % de odds); veterano frente a medio: {or_vet_vs_medio:.2f}")
print("Ejercicio 4 correcto ✔")

## Ejercicio 5 — La familia GLM: Binomial y Poisson

Con `datos` (ejercicio 2): ajusta con `smf.glm` y familia **Binomial** el mismo modelo del ejercicio 2 (`glm_bin`) y comprueba que los coeficientes coinciden con los de `modelo` (`dif_max`: la mayor diferencia absoluta). Después ajusta una regresión de **Poisson** para `tickets ~ gasto` (`glm_poisson`) y calcula `razon_tasas_10`: por cuánto se multiplica el número medio de tickets con 10 € más de gasto, `exp(10 × coeficiente)`.

In [ ]:
glm_bin = None        # TODO: smf.glm(formula, data=datos, family=sm.families.Binomial()).fit()
dif_max = None        # TODO
glm_poisson = None    # TODO: smf.glm("tickets ~ gasto", ..., family=sm.families.Poisson()).fit()
razon_tasas_10 = None # TODO

# Comprobación automática
assert all(v is not None for v in (glm_bin, dif_max, glm_poisson, razon_tasas_10)), "Completa las líneas marcadas con TODO"
assert dif_max < 1e-6, "un GLM Binomial con enlace logit debe dar los mismos coeficientes que smf.logit"
ref_po = smf.glm("tickets ~ gasto", data=datos, family=sm.families.Poisson()).fit()
assert np.isclose(razon_tasas_10, np.exp(10 * ref_po.params["gasto"])), "exp(10 * coeficiente de gasto) del modelo de Poisson"
assert 1.0 < razon_tasas_10 < 1.4, "con 10 € más de gasto, el número medio de tickets sube unos pocos puntos porcentuales"
print(f"dif_max = {dif_max:.1e}   razón de tasas por 10 €: {razon_tasas_10:.3f}")
print("Ejercicio 5 correcto ✔")

---
## Reto final — Ajusta e interpreta un modelo de abandono

Con 3 000 clientes nuevos (`datos_r`), ajusta el modelo logístico de `formula_r` (sin selección de variables ni VIF: eso es de la clase 12) y calcula: `modelo_r`; `tabla_or` (DataFrame con las columnas `"OR"`, `"IC_inf"` e `"IC_sup"`, una fila por variable y **sin** el Intercept); `p_perfil` (probabilidad prevista de un cliente de 12 meses, 45 €, 2 tickets y sin plan anual, como número); y `ame_tickets` (efecto marginal medio, en puntos porcentuales, de un ticket más). Escribe en `frases` una lista de 3 o 4 frases (cada una de al menos 40 caracteres) que lean correctamente los odds ratio de `tickets`, `plan_anual` y `gasto` (por 10 €): con "las demás variables fijas", en términos de **odds**, no de probabilidad.

In [ ]:
rng_r = np.random.default_rng(99)
datos_r = generar_clientes(3000, rng_r)
formula_r = "abandona ~ antiguedad + gasto + tickets + plan_anual"
perfil_r = pd.DataFrame({"antiguedad": [12], "gasto": [45], "tickets": [2], "plan_anual": [0]})

modelo_r = None     # TODO
tabla_or = None     # TODO: DataFrame con columnas "OR", "IC_inf", "IC_sup" (sin Intercept)
p_perfil = None     # TODO
ame_tickets = None  # TODO
frases = None       # TODO: lista de 3 o 4 cadenas de texto

# Comprobación automática
assert all(v is not None for v in (modelo_r, tabla_or, p_perfil, ame_tickets, frases)), "Completa las líneas marcadas con TODO"
ref_r = smf.logit(formula_r, data=datos_r).fit(disp=0)
assert np.allclose(modelo_r.params, ref_r.params), "ajusta smf.logit con formula_r y datos_r"
ref_ic = np.exp(ref_r.conf_int()).drop(index="Intercept")
assert list(tabla_or.columns) == ["OR", "IC_inf", "IC_sup"] and "Intercept" not in tabla_or.index, "columnas OR, IC_inf, IC_sup y sin la fila Intercept"
assert np.allclose(tabla_or["OR"], np.exp(ref_r.params).drop(index="Intercept")) and np.allclose(tabla_or["IC_inf"], ref_ic[0]) and np.allclose(tabla_or["IC_sup"], ref_ic[1]), "exp de los coeficientes y de los extremos del intervalo"
assert np.isclose(np.squeeze(np.asarray(p_perfil, dtype=float)), float(ref_r.predict(perfil_r).iloc[0])), "modelo_r.predict(perfil_r)"
d_mas = datos_r.copy(); d_mas["tickets"] += 1
assert np.isclose(ame_tickets, float((ref_r.predict(d_mas) - ref_r.predict(datos_r)).mean() * 100)), "AME: media del cambio de probabilidad de cada cliente, en puntos porcentuales"
assert isinstance(frases, list) and 3 <= len(frases) <= 4 and all(isinstance(f, str) and len(f) >= 40 for f in frases), "escribe 3 o 4 frases de al menos 40 caracteres cada una"
assert sum("odds" in f.lower() for f in frases) >= 2, "lee el odds ratio en términos de odds"
print(tabla_or.round(3))
print(f"Probabilidad del perfil: {p_perfil:.3f}   AME de un ticket: {ame_tickets:.1f} puntos")
print("Reto final correcto ✔")

---
## Cierre de la práctica

Has recorrido el ciclo completo de lectura de un modelo logístico: convertir entre probabilidad, odds y log-odds, ajustar e interpretar odds ratio con su intervalo, leer una variable categórica frente a su referencia, calcular probabilidades y efectos marginales, y situar la logística dentro de los GLM. Continúa con la clase 12: Multicolinealidad, selección de variables y validación.

---
## Soluciones (¡no mires antes de haberlo intentado!)

Cada celda siguiente contiene la solución de un ejercicio. Puedes ejecutarlas para comparar con tu resultado.

### Solución del ejercicio 1

In [ ]:
p = np.array([0.05, 0.20, 0.50, 0.90])
beta = 0.69
p0 = 0.20

odds = p / (1 - p)
logodds = np.log(odds)
p_de_vuelta = 1 / (1 + np.exp(-logodds))
razon_odds = np.exp(beta)
odds_nuevos = (p0 / (1 - p0)) * razon_odds
p_nueva = odds_nuevos / (1 + odds_nuevos)
prob_se_duplica = bool(np.isclose(p_nueva, 2 * p0, atol=0.02))

assert all(v is not None for v in (odds, logodds, p_de_vuelta, razon_odds, p_nueva, prob_se_duplica)), "Completa las líneas marcadas con TODO"
assert np.allclose(odds, p / (1 - p)), "odds = p / (1 - p)"
assert np.allclose(logodds, np.log(p / (1 - p))), "los log-odds son el logaritmo natural de los odds"
assert np.allclose(p_de_vuelta, p), "la sigmoide de los log-odds debe devolver p"
assert np.isclose(razon_odds, np.exp(0.69)), "razon_odds = exp(beta)"
assert np.isclose(p_nueva, 1 / 3, atol=0.005), "odds de p0 = 0,25; nuevos odds = 0,25 × razon_odds ≈ 0,5; p = odds / (1 + odds)"
assert prob_se_duplica is False, "los odds se duplican, pero la probabilidad pasa de 0,20 a 0,333, no a 0,40"
print(f"odds = {np.round(odds, 3)}   p_nueva = {p_nueva:.3f}")
print("Ejercicio 1 correcto ✔")

### Solución del ejercicio 2

In [ ]:
datos = generar_clientes(1500, np.random.default_rng(2026))
formula = "abandona ~ antiguedad + gasto + tickets + plan_anual"

modelo = smf.logit(formula, data=datos).fit(disp=0)
odds_ratios = np.exp(modelo.params)
ic_or = np.exp(modelo.conf_int())
or_tickets_10pct = (odds_ratios["tickets"] - 1) * 100

assert all(v is not None for v in (modelo, odds_ratios, ic_or, or_tickets_10pct)), "Completa las líneas marcadas con TODO"
ref = smf.logit(formula, data=datos).fit(disp=0)
assert np.allclose(modelo.params, ref.params), "ajusta smf.logit con la fórmula y los datos indicados"
assert np.allclose(odds_ratios, np.exp(ref.params)), "odds ratio = exp de los coeficientes"
assert np.allclose(ic_or.values, np.exp(ref.conf_int()).values), "aplica exp a los extremos del intervalo, no al ancho"
assert np.isclose(or_tickets_10pct, (np.exp(ref.params["tickets"]) - 1) * 100), "(exp(coef de tickets) − 1) × 100"
for v in ("tickets", "plan_anual", "antiguedad"):
    assert abs(modelo.params[v] - VERDAD[v]) < 4 * modelo.bse[v], f"el coeficiente estimado de {v} debería estar cerca del verdadero"
print(f"Cada ticket multiplica los odds por {odds_ratios['tickets']:.2f} (+{or_tickets_10pct:.0f} %)")
print("Ejercicio 2 correcto ✔")

### Solución del ejercicio 3

In [ ]:
modelo3 = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=datos).fit(disp=0)
perfil_medio = pd.DataFrame({"antiguedad": [24], "gasto": [40], "tickets": [1], "plan_anual": [0]})
perfil_bajo = pd.DataFrame({"antiguedad": [72], "gasto": [20], "tickets": [0], "plan_anual": [1]})

def con_un_ticket_mas(perfil):
    nuevo = perfil.copy()
    nuevo["tickets"] = nuevo["tickets"] + 1
    return nuevo

cambio_medio = float((modelo3.predict(con_un_ticket_mas(perfil_medio)) - modelo3.predict(perfil_medio)).iloc[0] * 100)
cambio_bajo = float((modelo3.predict(con_un_ticket_mas(perfil_bajo)) - modelo3.predict(perfil_bajo)).iloc[0] * 100)
ame_pp = float((modelo3.predict(con_un_ticket_mas(datos)) - modelo3.predict(datos)).mean() * 100)
mayor_en_medio = bool(cambio_medio > cambio_bajo)

assert all(v is not None for v in (cambio_medio, cambio_bajo, ame_pp, mayor_en_medio)), "Completa las líneas marcadas con TODO"
ref_m = float((modelo3.predict(con_un_ticket_mas(perfil_medio)) - modelo3.predict(perfil_medio)).iloc[0] * 100)
ref_b = float((modelo3.predict(con_un_ticket_mas(perfil_bajo)) - modelo3.predict(perfil_bajo)).iloc[0] * 100)
ref_ame = float((modelo3.predict(con_un_ticket_mas(datos)) - modelo3.predict(datos)).mean() * 100)
assert np.isclose(np.squeeze(np.asarray(cambio_medio, dtype=float)), ref_m), "el cambio se mide en puntos porcentuales: (p después − p antes) × 100"
assert np.isclose(np.squeeze(np.asarray(cambio_bajo, dtype=float)), ref_b), "usa perfil_bajo con la función con_un_ticket_mas"
assert np.isclose(ame_pp, ref_ame), "el AME promedia el cambio de probabilidad de CADA cliente de datos"
assert bool(mayor_en_medio) is True, "compara cambio_medio con cambio_bajo: la probabilidad de partida baja deja poco margen"
print(f"Cambio con +1 ticket: perfil medio {ref_m:.1f} puntos, perfil bajo {ref_b:.1f} puntos, AME {ref_ame:.1f} puntos")
print("Ejercicio 3 correcto ✔")

### Solución del ejercicio 4

In [ ]:
datos["tramo"] = pd.cut(datos["antiguedad"], [0, 11, 35, 72], labels=["nuevo", "medio", "veterano"]).astype(str)
f_tramo = "abandona ~ C(tramo, Treatment(reference='nuevo')) + gasto + tickets + plan_anual"
n_vet = "C(tramo, Treatment(reference='nuevo'))[T.veterano]"
n_med = "C(tramo, Treatment(reference='nuevo'))[T.medio]"

m_tramo = smf.logit(f_tramo, data=datos).fit(disp=0)
or_veterano = np.exp(m_tramo.params[n_vet])
pct_menos_odds = (1 - or_veterano) * 100
or_vet_vs_medio = np.exp(m_tramo.params[n_vet] - m_tramo.params[n_med])

assert all(v is not None for v in (m_tramo, or_veterano, pct_menos_odds, or_vet_vs_medio)), "Completa las líneas marcadas con TODO"
ref = smf.logit(f_tramo, data=datos).fit(disp=0)
assert np.allclose(m_tramo.params, ref.params), "ajusta smf.logit con f_tramo y los datos"
assert np.isclose(or_veterano, np.exp(ref.params[n_vet])), "or_veterano = np.exp(m_tramo.params[n_vet])"
assert or_veterano < 1, "los clientes veteranos abandonan menos que los nuevos: el odds ratio es menor que 1"
assert np.isclose(pct_menos_odds, (1 - np.exp(ref.params[n_vet])) * 100), "(1 − odds ratio) × 100"
assert np.isclose(or_vet_vs_medio, np.exp(ref.params[n_vet] - ref.params[n_med])), "veterano frente a medio: exp(b_veterano − b_medio), o el cociente de los dos odds ratio"
print(f"Odds ratio veterano frente a nuevo: {or_veterano:.2f} (−{pct_menos_odds:.0f} % de odds); veterano frente a medio: {or_vet_vs_medio:.2f}")
print("Ejercicio 4 correcto ✔")

### Solución del ejercicio 5

In [ ]:
glm_bin = smf.glm(formula, data=datos, family=sm.families.Binomial()).fit()
dif_max = float(np.abs(glm_bin.params - modelo.params).max())
glm_poisson = smf.glm("tickets ~ gasto", data=datos, family=sm.families.Poisson()).fit()
razon_tasas_10 = float(np.exp(10 * glm_poisson.params["gasto"]))

assert all(v is not None for v in (glm_bin, dif_max, glm_poisson, razon_tasas_10)), "Completa las líneas marcadas con TODO"
assert dif_max < 1e-6, "un GLM Binomial con enlace logit debe dar los mismos coeficientes que smf.logit"
ref_po = smf.glm("tickets ~ gasto", data=datos, family=sm.families.Poisson()).fit()
assert np.isclose(razon_tasas_10, np.exp(10 * ref_po.params["gasto"])), "exp(10 * coeficiente de gasto) del modelo de Poisson"
assert 1.0 < razon_tasas_10 < 1.4, "con 10 € más de gasto, el número medio de tickets sube unos pocos puntos porcentuales"
print(f"dif_max = {dif_max:.1e}   razón de tasas por 10 €: {razon_tasas_10:.3f}")
print("Ejercicio 5 correcto ✔")

### Solución del reto final

In [ ]:
rng_r = np.random.default_rng(99)
datos_r = generar_clientes(3000, rng_r)
formula_r = "abandona ~ antiguedad + gasto + tickets + plan_anual"
perfil_r = pd.DataFrame({"antiguedad": [12], "gasto": [45], "tickets": [2], "plan_anual": [0]})

modelo_r = smf.logit(formula_r, data=datos_r).fit(disp=0)
ic_r = np.exp(modelo_r.conf_int())
tabla_or = pd.DataFrame({"OR": np.exp(modelo_r.params), "IC_inf": ic_r[0], "IC_sup": ic_r[1]}).drop(index="Intercept")
p_perfil = float(modelo_r.predict(perfil_r).iloc[0])
datos_mas = datos_r.copy()
datos_mas["tickets"] = datos_mas["tickets"] + 1
ame_tickets = float((modelo_r.predict(datos_mas) - modelo_r.predict(datos_r)).mean() * 100)
frases = [
    "Con las demás variables fijas, cada ticket adicional multiplica los odds de abandono por el odds ratio de tickets (mayor que 1), no la probabilidad.",
    "Tener plan anual multiplica los odds de abandono por un odds ratio menor que 1: los reduce, con las demás variables fijas.",
    "Diez euros más de gasto mensual multiplican los odds por exp(10 × coeficiente de gasto), un aumento moderado de los odds."]

assert all(v is not None for v in (modelo_r, tabla_or, p_perfil, ame_tickets, frases)), "Completa las líneas marcadas con TODO"
ref_r = smf.logit(formula_r, data=datos_r).fit(disp=0)
assert np.allclose(modelo_r.params, ref_r.params), "ajusta smf.logit con formula_r y datos_r"
ref_ic = np.exp(ref_r.conf_int()).drop(index="Intercept")
assert list(tabla_or.columns) == ["OR", "IC_inf", "IC_sup"] and "Intercept" not in tabla_or.index, "columnas OR, IC_inf, IC_sup y sin la fila Intercept"
assert np.allclose(tabla_or["OR"], np.exp(ref_r.params).drop(index="Intercept")) and np.allclose(tabla_or["IC_inf"], ref_ic[0]) and np.allclose(tabla_or["IC_sup"], ref_ic[1]), "exp de los coeficientes y de los extremos del intervalo"
assert np.isclose(np.squeeze(np.asarray(p_perfil, dtype=float)), float(ref_r.predict(perfil_r).iloc[0])), "modelo_r.predict(perfil_r)"
d_mas = datos_r.copy(); d_mas["tickets"] += 1
assert np.isclose(ame_tickets, float((ref_r.predict(d_mas) - ref_r.predict(datos_r)).mean() * 100)), "AME: media del cambio de probabilidad de cada cliente, en puntos porcentuales"
assert isinstance(frases, list) and 3 <= len(frases) <= 4 and all(isinstance(f, str) and len(f) >= 40 for f in frases), "escribe 3 o 4 frases de al menos 40 caracteres cada una"
assert sum("odds" in f.lower() for f in frases) >= 2, "lee el odds ratio en términos de odds"
print(tabla_or.round(3))
print(f"Probabilidad del perfil: {p_perfil:.3f}   AME de un ticket: {ame_tickets:.1f} puntos")
print("Reto final correcto ✔")

---
## Autoevaluación

Anota en una celda de texto o en papel:

- **¿Qué ejercicios resolví sin ayuda?**
- **¿Qué concepto sigue poco claro?** (por ejemplo: odds ratio frente a probabilidad, categoría de referencia, efecto marginal, GLM)
- **Tiempo dedicado:** ___ minutos
- **Una pregunta que quiero llevar a la próxima clase:** ...